# =============================== # Modelling Notebook # ===============================

## Objectives

- Fit and evaluate regression models to predict SalePrice.
- Apply feature engineering pipeline developed earlier.
- Compare candidate regression models using cross-validation.
- Select best model and optimise hyperparameters.
- Export final train/test sets, pipelines, and feature importance plot.


## Inputs 
- outputs/datasets/processed/TrainSet.csv
- outputs/datasets/processed/TestSet.csv

## Outputs
- Train set (features and target)
- Test set (features and target)
- Modeling pipeline
- Feature importance plot

### Import Cell

In [1]:
import sys
!"{sys.executable}" -m pip install xgboost


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.model_selection import train_test_split

# Pipelines
from sklearn.pipeline import Pipeline

# Feature Scaling
from sklearn.preprocessing import StandardScaler

# Feature Selection
from sklearn.feature_selection import SelectFromModel

# Regression Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from xgboost import XGBRegressor

# Hyperparameter Search
from sklearn.model_selection import GridSearchCV

# Metrics
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

---

## Change working directory

We need to set the current working directory to the parent folder for consistency.

In [3]:
# Check current directory
current_dir = os.getcwd()
print("Current directory:", current_dir)

# Move to parent directory
os.chdir(os.path.dirname(current_dir))
print("New current directory:", os.getcwd())

Current directory: c:\Users\david\Portfolio 5\heritage-housing\jupyter_notebooks
New current directory: c:\Users\david\Portfolio 5\heritage-housing


Confirm the new current directory

In [4]:
current_dir = os.getcwd()
current_dir

'c:\\Users\\david\\Portfolio 5\\heritage-housing'

## Load Cleaned Data and Split into Trained and Test Sets



In [5]:
BASE_DIR = Path(r"C:\Users\david\Portfolio 5\heritage-housing")
data_path = BASE_DIR / "outputs/datasets/cleaned/CleanedData.csv"

# Load data
train_path = BASE_DIR / "outputs/datasets/processed/TrainSet.csv"
test_path = BASE_DIR / "outputs/datasets/processed/TestSet.csv"

TrainSet = pd.read_csv(train_path)
TestSet = pd.read_csv(test_path)

target = "SalePrice"

X_train = TrainSet.drop(columns=[target])
y_train = TrainSet[target]

X_test = TestSet.drop(columns=[target])
y_test = TestSet[target]

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (1168, 30)
Test shape: (292, 30)


---

## Helper Functions

These functions help check missing values and generate diagnostic plots for numeric and categorical features.

In [6]:
def evaluate_regression (x, y, pipeline):
    """
    Evaluates a trained regression model using common performance metrics.
    """
    y_pred = pipeline.predict(x)

    mae = mean_absolute_error(y, y_pred)
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    r2 = r2_score(y, y_pred)

    print("MAE:", mae)
    print("RMSE:", rmse)
    print("R2:", r2)

    return mae, rmse, r2

def plot_predictions(x, y, pipeline):
    """
    Plots actual vs predicted values for regression model evaluation.
    """
    y_pred = pipeline.predict(x)

    plt.figure(figsize=(6,6))
    plt.scatter(y, y_pred)
    plt.xlabel("Actual Values")
    plt.ylabel("Predicted Values")
    plt.title("Actual vs Predicted")
    plt.show()

def plot_residuals(x, y, pipeline):
    """
    Plots residuals (errors) of a regression model to assess model fit.
    Residuals are calculated as: actual - predicted.
    """
    y_pred = pipeline.predict(x)
    residuals = y - y_pred

    plt.figure(figsize=(6,4))
    sns.histplot(residuals, kde= True)
    plt.xlabel("Residuals (Actual - Predicted)")
    plt.title("Residual Distribution")
    plt.show()

def save_model_outputs(model, file_path):
    """
    Saves a trained machine learning pipeline to a specified directory.
    """

    os.makedirs(file_path, exist_ok=True)

    joblib.dump(model, f"{file_path}/model.pkl")

    print(f"Model successfully saved to: {file_path}")

# ===============================
## Model Pipeline
# ===============================

### Model Benchmarking

#### Define Candidate Models

In [7]:
models_quick_search = {
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "RandomForestRegressor": RandomForestRegressor(random_state=42),
    "GradientBoostingRegressor": GradientBoostingRegressor(random_state=42),
    "ExtraTreesRegressor": ExtraTreesRegressor(random_state=42),
    "XGBRegressor": XGBRegressor(random_state=42)
}

### Define Hyperparameter Grids (Baseline)

In [8]:
params_quick_search = {
    "LinearRegression": {},
    "Ridge": {},
    "Lasso": {},
    "RandomForestRegressor": {},
    "GradientBoostingRegressor": {},
    "ExtraTreesRegressor": {},
    "XGBRegressor": {}
}

### Run GridSearchCV (Quick Benchmark)

#### Custom class for Hyperparameter Optimization

In [9]:
def PipelineReg(model):
    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("feat_selection", SelectFromModel(model)),
        ("model", model),
    ])
    return pipeline


class HyperparameterOptimizationSearch:

    def __init__(self, models, params):
        self.models = models
        self.params = params
        self.keys = models.keys()
        self.grid_searches = {}

    def fit(self, x, y, cv, n_jobs, verbose=1, scoring=None):
        for key in self.keys:
            print(f"\nRunning GridSearchCV for {key}\n")

            model = PipelineReg(self.models[key])
            params = self.params[key]

            gs = GridSearchCV(
                estimator=model,
                param_grid=params,
                cv=cv,
                n_jobs=n_jobs,
                verbose=verbose,
                scoring=scoring
            )

            gs.fit(x, y)
            self.grid_searches[key] = gs

    def score_summary(self, sort_by='mean_score'):

        def row(key, scores, params):
            d = {
                'estimator': key,
                'min_score': min(scores),
                'max_score': max(scores),
                'mean_score': np.mean(scores),
                'std_score': np.std(scores),
            }
            return pd.Series({**params, **d})

        rows = []
        for k in self.grid_searches:
            params = self.grid_searches[k].cv_results_['params']
            scores = []

            for i in range(self.grid_searches[k].cv):
                key = f"split{i}_test_score"
                r = self.grid_searches[k].cv_results_[key]
                scores.append(r.reshape(len(params), 1))

            all_scores = np.hstack(scores)

            for p, s in zip(params, all_scores):
                rows.append(row(k, s, p))

        df = pd.concat(rows, axis=1).T.sort_values([sort_by], ascending=False)

        columns = ['estimator', 'min_score', 'mean_score', 'max_score', 'std_score']
        columns = columns + [c for c in df.columns if c not in columns]

        return df[columns], self.grid_searches

In [10]:
print(X_train.select_dtypes(include="object").columns)

Index([], dtype='object')


In [11]:
search = HyperparameterOptimizationSearch(
    models=models_quick_search,
    params=params_quick_search
)

search.fit(
    X_train,
    y_train,
    scoring="neg_mean_absolute_error",
    cv=5,
    n_jobs=-1
)


Running GridSearchCV for LinearRegression

Fitting 5 folds for each of 1 candidates, totalling 5 fits

Running GridSearchCV for Ridge

Fitting 5 folds for each of 1 candidates, totalling 5 fits

Running GridSearchCV for Lasso

Fitting 5 folds for each of 1 candidates, totalling 5 fits


c:\Users\david\Portfolio 5\heritage-housing\venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.344e+09, tolerance: 6.967e+08
  model = cd_fast.enet_coordinate_descent(
c:\Users\david\Portfolio 5\heritage-housing\venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 5.344e+09, tolerance: 6.967e+08
  model = cd_fast.enet_coordinate_descent(



Running GridSearchCV for RandomForestRegressor

Fitting 5 folds for each of 1 candidates, totalling 5 fits

Running GridSearchCV for GradientBoostingRegressor

Fitting 5 folds for each of 1 candidates, totalling 5 fits

Running GridSearchCV for ExtraTreesRegressor

Fitting 5 folds for each of 1 candidates, totalling 5 fits

Running GridSearchCV for XGBRegressor

Fitting 5 folds for each of 1 candidates, totalling 5 fits


### Evaluate Model Performance

In [12]:
grid_search_summary, grid_search_pipelines = search.score_summary(sort_by='mean_score')

grid_search_summary

,estimator,min_score,mean_score,max_score,std_score
4,GradientBoostingRegressor,-23198.373271,-21007.339918,-18875.348845,1741.276318
3,RandomForestRegressor,-24629.738226,-21326.009354,-18973.036738,2128.121429
2,Lasso,-24105.986735,-21971.710865,-20237.648381,1441.511813
5,ExtraTreesRegressor,-24603.123611,-22295.876948,-20084.545107,1548.986757
1,Ridge,-25415.642876,-23065.884404,-20572.199669,1714.768219
0,LinearRegression,-25423.272546,-23073.526529,-20577.636221,1714.988218
6,XGBRegressor,-28262.935547,-24461.834766,-21418.595703,2600.712291


#### Model Selection for Hyperparameter Optimisation

Based on baseline model evaluation using cross-validation, GradientBoostingRegressor achieved the best performance (lowest MAE and good stability across folds).

Therefore, it is selected as the final model for hyperparameter tuning.

### Select Best Performing Model

In [13]:
best_model = grid_search_summary.iloc[0, 0]
best_model

'GradientBoostingRegressor'

### Define Hyperparameter Grid for Best Model

In [ ]:
models_search = {
    "GradientBoostingRegressor": GradientBoostingRegressor(random_state=42)
}

params_search = {
    "GradientBoostingRegressor": {
        "model__n_estimators": [100, 300],
        "model__learning_rate": [0.1, 0.01, 0.001],
        "model__max_depth": [3, 5, 10]
    }
}

### Run GridSearchCV (Tuned Model)

In [15]:
search = HyperparameterOptimizationSearch(
    models=models_search,
    params=params_search
)

search.fit(
    X_train,
    y_train,
    scoring="neg_mean_absolute_error",
    cv=5,
    n_jobs=-1
)


Running GridSearchCV for GradientBoostingRegressor

Fitting 5 folds for each of 324 candidates, totalling 1620 fits


KeyboardInterrupt: 

### Evaluate Tuned Model Performance

In [ ]:
grid_search_summary, grid_search_pipelines = search.score_summary(sort_by='mean_score')

grid_search_summary

### Extract Best Model Pipeline

In [ ]:
best_model_name = grid_search_summary.iloc[0, 0]
pipeline_reg = grid_search_pipelines[best_model_name].best_estimator_

pipeline_reg

## Model Evaluation 

### Evaluate Model on Train Set

In [ ]:
evaluate_regression(X_train, y_train, pipeline_reg)

### Evaluate Model on Test Set


In [ ]:
evaluate_regression(X_test, y_test, pipeline_reg)

#### Model Train Set vs Test Set comparison 
The model achieves strong performance (R²: 0.93 train, 0.82 test) with mild overfitting. Test MAE (~£22k) indicates reasonable prediction error, with higher errors likely on extreme values.

---

### Plot Actual vs Predicted Values

In [ ]:
plot_predictions(X_test, y_test, pipeline_reg)

### Plot Residual Distribution

In [ ]:
plot_residuals(X_test, y_test, pipeline_reg)

#### Model Fit: Actual vs Predicted Analysis

The model demonstrates strong overall fit, with predictions closely tracking actual values. However, errors increase for higher-priced properties, indicating reduced performance on extreme values.

## Feature Importance

### Extract Feature Importance from the Model

In [ ]:
model = pipeline_reg.named_steps["model"]
mask = pipeline_reg.named_steps["feat_selection"].get_support()
selected_features = X_train.columns[mask]

df_feature_importance = pd.DataFrame({
    "Feature": selected_features,
    "Importance": model.feature_importances_
}).sort_values(by="Importance", ascending=False)

### Plot Feature Importance

---

In [ ]:
plt.figure(figsize=(12, 6))
df_feature_importance.plot(kind="bar", x="Feature", y="Importance", legend=False)
plt.title("Feature Importance")
plt.ylabel("Importance Score")
plt.tight_layout()
plt.show()

#### Feature Importance Summary

GrLivArea is clearly the dominant predictor, followed by YearBuilt, with TotalBsmtSF, GarageArea, and BsmtFinSF1 contributing moderately. KitchenQual_Ex has minimal impact, indicating the model is driven mainly by size and construction features.


